In [0]:
spark.sql("USE CATALOG f1_catalog")
spark.sql("USE SCHEMA silver")

###circuits

In [0]:
circuit_df = spark.table("f1_catalog.bronze.circuits")
circuit_df=circuit_df.withColumnRenamed("circuitId","circuit_id").withColumnRenamed("circuitRef","circuit_ref")
#remove duplicates
circuit_df=circuit_df.dropDuplicates(["circuit_id"])

In [0]:
from pyspark.sql.functions import *
#remove null values
circuits_df=circuit_df.filter(col("circuit_id").isNotNull())

In [0]:
from pyspark.sql.functions import current_timestamp

circuits_df = circuits_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
circuits_df.write.format('delta').mode('overwrite').saveAsTable('f1_catalog.silver.circuits')

###constructors

In [0]:
constructor_df = spark.table('f1_catalog.bronze.constructors')
#rename column
constructor_df=constructor_df.withColumnRenamed("constructorId",'constructor_id').withColumnRenamed("constructorRef","constructor_ref")
#remove duplicates
constructor_df=constructor_df.dropDuplicates(["constructor_id"])

In [0]:
from pyspark.sql.functions import col
#filter invalid records
constructor_df=constructor_df.filter(col('constructor_id').isNotNull())

In [0]:
from pyspark.sql.functions import *
constructor_df = constructor_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
constructor_df.write.format('delta').mode('overwrite').option("overwriteSchema", "true").saveAsTable('f1_catalog.silver.constructor')

###drivers

In [0]:
drivers_df = spark.table("f1_catalog.bronze.drivers")

In [0]:
from pyspark.sql.functions import concat, col

drivers_df = drivers_df.withColumn(
    "driver_name",
    concat_ws(" ",col("name.forename"), col("name.surname"))
)

In [0]:
drivers_df = drivers_df.withColumnRenamed("driverId","driver_id").withColumnRenamed("driverRef","driver_ref")

#taking only useful column
drivers_df = drivers_df.select(
    "driver_id",
    "driver_ref",
    "number",
    "code",
    "driver_name",
    "dob",
    "nationality"
)

In [0]:
#remove duplicates
drivers_df = drivers_df.dropDuplicates(["driver_id"])

#filter null records
drivers_df = drivers_df.filter(col("driver_id").isNotNull())

In [0]:
from pyspark.sql.functions import current_timestamp

drivers_df = drivers_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
drivers_df.write.format("delta").mode("overwrite").saveAsTable("f1_catalog.silver.drivers")

###races

In [0]:
races_df=spark.table('f1_catalog.bronze.races')

In [0]:
races_df=races_df.withColumnRenamed("raceId","race_id").withColumnRenamed("circuitId","circuit_id")

In [0]:
# create timestamp 
from pyspark.sql.functions import expr
races_df = races_df.withColumn(

    "race_timestamp",
    expr("try_to_timestamp(concat(date,' ',time))")
)

In [0]:
# remove duplicates
races_df = races_df.dropDuplicates(["race_id"])

In [0]:
from pyspark.sql.functions import to_timestamp, concat, col, lit, current_timestamp

# add metadata
races_df = races_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
# write silver table
races_df.write.format("delta").mode("overwrite").saveAsTable("f1_catalog.silver.races")

In [0]:
display(races_df.select("date","time","race_timestamp"))

###results

In [0]:
results_df=spark.table('f1_catalog.bronze.results')

In [0]:
results_df=results_df.withColumnRenamed("resultId","result_id") \
.withColumnRenamed("raceId","race_id") \
.withColumnRenamed("driverId","driver_id") \
.withColumnRenamed("constructorId","constructor_id")

In [0]:
from pyspark.sql.functions import *
results_df = results_df.filter(
    col("result_id").isNotNull() &
    col("race_id").isNotNull() &
    col("driver_id").isNotNull() &
    col("constructor_id").isNotNull()
)

results_df = results_df.dropDuplicates(["race_id", "driver_id"])

In [0]:
results_df = results_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
results_df.write.format('delta').mode('overwrite').saveAsTable('f1_catalog.silver.results')

###pitstops

In [0]:
pitstops_df=spark.table('f1_catalog.bronze.pit_stops')

In [0]:
#rename column
pitstops_df=pitstops_df.withColumnRenamed("raceId",'race_id').withColumnRenamed("driverId",'driver_id')

In [0]:
from pyspark.sql.functions import *

pitstops_df = pitstops_df.filter(
    col("race_id").isNotNull() &
    col("driver_id").isNotNull() &
    col("stop").isNotNull() &
    col("duration").isNotNull()
)

pitstops_df = pitstops_df.dropDuplicates(
    ["race_id", "driver_id", "stop"]
)

In [0]:
pitstops_df=pitstops_df.withColumn("ingestion_date",current_timestamp())

In [0]:
pitstops_df.write.mode("overwrite").format('delta').saveAsTable('f1_catalog.silver.pitstops')

###laptimes

In [0]:
lap_df = spark.table("f1_catalog.bronze.lap_times")

In [0]:
#rename column
lap_df=lap_df.withColumnRenamed('raceId','race_id').withColumnRenamed('driverId','driver_id')

In [0]:

lap_df = lap_df.filter(
    col("race_id").isNotNull() &
    col("driver_id").isNotNull() &
    col("lap").isNotNull() &
    col("position").isNotNull() &
    col("time").isNotNull()
)

lap_df = lap_df.dropDuplicates(
    ["race_id", "driver_id", "lap"]
)

In [0]:
lap_df = lap_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
lap_df.write.format('delta').mode('overwrite').saveAsTable('f1_catalog.silver.laptimes')

###qualifying

In [0]:
qualifying_df=spark.table('f1_catalog.bronze.qualifying')

In [0]:
#rename column
qualifying_df=qualifying_df.withColumnRenamed("qualifyId","qualify_id").withColumnRenamed("raceId","race_id").withColumnRenamed("driverId","driver_id").withColumnRenamed("constructorId","constructor_id")



In [0]:
qualifying_df = qualifying_df.filter(
    col("qualify_id").isNotNull() &
    col("race_id").isNotNull() &
    col("driver_id").isNotNull() &
    col("constructor_id").isNotNull()
)

In [0]:
qualifying_df = qualifying_df.dropDuplicates(
    ["race_id", "driver_id"]
)

In [0]:
from pyspark.sql.functions import current_timestamp
qualifying_df = qualifying_df.withColumn(
    "ingestion_date",
    current_timestamp()
)

In [0]:
qualifying_df.write.format("delta").mode('overwrite').saveAsTable("f1_catalog.silver.qualifying")